# CertGen ICML 2027 — cifar_10k_generation
Planning/execution notebook. `claim_allowed=false`. No outputs are source-controlled.


In [ ]:
# Trusted bootstrap and exact package identity
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
from certgen.notebooks.trusted_bootstrap import discover_authenticated_package
EXPECTED_IDENTITY = json.loads(os.environ['CERTGEN_EXPECTED_PACKAGE_IDENTITY_JSON'])
INPUT_ROOTS = [Path('/kaggle/input')]
authenticated = discover_authenticated_package(INPUT_ROOTS, EXPECTED_IDENTITY)
assert authenticated['selection_status'] in {'SELECTED_UNIQUE_VALID_PACKAGE','DUPLICATE_IDENTICAL_COPY_DEDUPED'}


In [ ]:
# Dependency restart marker, GPU visibility, worker isolation, and disk guard
RESTART_MARKER = Path('/kaggle/working/.certgen_dependency_restart_complete')
if not RESTART_MARKER.exists():
    raise RuntimeError('dependency bootstrap/restart marker is required')
import torch
if torch.cuda.device_count() != 2:
    raise RuntimeError(f'exactly two visible GPUs required, found {torch.cuda.device_count()}')
free = shutil.disk_usage('/kaggle/working').free
if free < 10 * 1024**3:
    raise RuntimeError('disk guard: fewer than 10 GiB free')
WORKER_ENV = {'CUDA_VISIBLE_DEVICES': None, 'CERTGEN_CPU_ONLY': '0'}


In [ ]:
# Deterministic generation shards, asset resolution, resume/restart, and output identity closure
STAGE = 'generation'
NOTEBOOK_ID = 'cifar_10k_generation'
SHARDS = list(range(int(os.environ.get('CERTGEN_NUM_SHARDS', '1'))))
for shard_id in SHARDS:
    marker = Path(f'/kaggle/working/completed_shard_{shard_id:04d}.json')
    if marker.exists():
        continue  # deterministic resume
    raise RuntimeError('worker execution requires an authenticated stage input and explicit worker command')
# A real run replaces the fail-closed boundary above only through the source-controlled worker.
# Final ZIP validation must verify membership, hashes, stage identity, configuration hash, and completion status.
# Copy the validated ZIP back before local import/resume; never treat notebook state as evidence.
